# RL Vectorizer — Colab SFT Training

In [ ]:
!git clone https://github.com/dersteppenwolfruowen-316/RL-Vec.git
%cd /content/RL-Vec
!pip install -q torch transformers peft accelerate bitsandbytes datasets
!pip install -q lxml cairosvg pillow scikit-image opencv-python tensorboard
!pip install -q hydra-core omegaconf shapely

In [ ]:
import urllib.request, zipfile, os
%cd /content/RL-Vec
DATA_DIR = "data/resplan"
os.makedirs(DATA_DIR, exist_ok=True)
ZIP = os.path.join(DATA_DIR, "ResPlan.zip")
if not os.path.exists(ZIP):
    print("Downloading...")
    urllib.request.urlretrieve("https://github.com/m-agour/ResPlan/releases/download/1.0.0/ResPlan.zip", ZIP)
if not os.path.exists(os.path.join(DATA_DIR, "ResPlan.pkl")):
    with zipfile.ZipFile(ZIP) as zf: zf.extractall(DATA_DIR)
if not os.path.exists(os.path.join(DATA_DIR, "svgs")):
    !python convert_resplan.py
if not os.path.exists(os.path.join(DATA_DIR, "sft_train.jsonl")):
    !python scripts/prepare_sft_data.py
print("Ready!")

In [ ]:
%cd /content/RL-Vec
!python scripts/train_sft.py --max-samples 200 --batch-size 1 --epochs 3 --lr 1e-4 --save-dir checkpoints/sft --log-interval 5

In [ ]:
%cd /content/RL-Vec
import torch
from PIL import Image
from transformers import AutoProcessor
from peft import PeftModel
from transformers import Qwen2_5_VLForConditionalGeneration

model = Qwen2_5_VLForConditionalGeneration.from_pretrained("Qwen/Qwen2.5-VL-3B-Instruct", torch_dtype=torch.bfloat16, device_map="auto")
model = PeftModel.from_pretrained(model, "checkpoints/sft/final")
processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-3B-Instruct", use_fast=False)

img = Image.open("data/resplan/bitmaps/resplan_00013.png").convert("RGB")
msg = [{"role": "user", "content": [{"type": "image", "image": img}, {"type": "text", "text": "Convert this floor plan to SVG."}]}]
txt = processor.apply_chat_template(msg, tokenize=False, add_generation_prompt=True)
inp = processor(text=[txt], images=[img], return_tensors="pt").to("cuda")

with torch.no_grad():
    gen = model.generate(**inp, max_new_tokens=1024)
    out = processor.decode(gen[0][inp["input_ids"].shape[1]:], skip_special_tokens=True)
print(out[:500])